<a href="https://colab.research.google.com/github/AIVIETNAM-AIO-HUYTRUONG/AIO-2026/blob/Tree-based-Algorithms/M3/ML-Base/Tree-based%20Algorithms/Step_by_Step_Decision_Tree_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Decision Tree là gì?

- Hãy tưởng tượng chúng ta đang vẽ một sơ đồ: Khởi đầu bằng một câu hỏi duy nhất, ví dụ mỗi khi ra khỏi nhà, ta phân vân có nên mang ô theo không. Lúc này, ta sẽ xem xét tình hình thời tiết với câu hỏi: "Trời có đang mưa không?". Mỗi khi trả lời "có" hoặc "không", ta sẽ rẽ nhánh tiếp theo kèm với câu hỏi mới, và tiếp tục như vậy cho đến khi ta chốt được quyết định cuối cùng.

- **Decision Tree** chính là mô hình máy học mô phỏng quy trình đó.
  - Khi cây dùng để phân loại dữ liệu thành các nhãn rời rạc, ta gọi đó là **Classification Tree (Cây phân loại)**.
  - Nếu dùng để dự đoán giá trị số liên tục, ta gọi **Regression Tree (Cây hồi quy)**.

  - Trong bài này, chúng ta sẽ tập trung tìm hiểu về **Classification Tree**.

<p align="center">
  <img src="../Images/Decision-tree.png" alt="Decision-tree"/>
</p>

- Decision Tree có thể xử lý đồng thời ba dạng dữ liệu:
  - **Dữ liệu rời rạc (Categorical data)**: Với các biến mang giá trị hữu hạn như **Đúng/Sai**, cây sẽ tạo nhánh tương ứng cho từng giá trị.
  
  - **Dữ liệu liên tục (Continuous/Numeric data)**: Với các biến số thực như **Tuổi/Điểm**. Thuật toán sẽ tìm **ngưỡng (threshold)** tối ưu.

  - **Dữ liệu hỗn hợp (Mixed data)**: Khi tập dữ liệu chứa cả dữ liệu rời rạc và dữ liệu liên tục, Decision Tree có thể kết hợp cả hai.

## 1. Giải quyết bài toán (split condition)

- Khi xây dựng **Decision Tree**, tại mỗi **nút (node)**, thuật toán phải đối mặt với một câu hỏi:
  > Trong số hàng chục **đặc trưng (features)** của dữ liệu, đâu là feature tốt nhất để dùng làm điều kiện phân chia nhánh?

- Để máy tính có thể tự động và định lượng được feature nào tốt, chúng ta cần một thước đo toán học. Đó chính là lý do **Entropy** và **Information Gain** ra đời.

## 2. Entropy & Information Gain

- Khái niệm **Entropy** được Claude Shannon đưa ra năm 1948 trong lý thuyết Thông tin (Information Theory), dùng để đo mức độ **không chắc chắn (uncertainty)** hoặc **độ xáo trộn/không thuần khiết (impurity)** của một tập dữ liệu.

- Trong một sự kiện $E$:
  - Một chiếc túi trong đó có
    - 9 viên bi đỏ: $E_{đỏ}$
    - 1 viên bi xanh: $E_{xanh}$
  - Xác suất của $E_{đỏ}$: $P({\text{đỏ}}) = \frac{9}{10} = 0.9$
  - Xác suất của $E_{xanh}$: $P({\text{xanh}}) = \frac{1}{10} = 0.1$

- Vì xác suất $P({\text{xanh}})$ nhỏ hơn $P({\text{đỏ}})$, ta thấy độ ngạc nhiên hơn khi rút được viên bi xanh so với viên bi đỏ. Vậy làm thế nào để đo lường "độ ngạc nhiên" này?

- **Đề xuất ban đầu**: $$\text{Surprise}(E) = \frac{1}{P(E)}$$

- Hàm này thỏa điều kiện:
  - Sự kiện càng hiếm $P(E) \downarrow$ thì độ ngạc nhiên lại càng cao $\text{Surprise}(E) \uparrow$.
  - Tuy nhiên, tồn tại một số vấn đề:
    1. **Đơn vị mơ hồ**: $P(E) = 0.1 \Rightarrow \text{Surprise} = 10$. Vậy **10** này là gì? Ta không thể nói là "10 ngạc nhiên".
    2. **Không cộng dồn được**: Với hai sự kiện độc lập $E_1, E_2$:
        - **Ý tưởng ban đầu (Mong muốn "Tính cộng dồn")**: Lượng bất ngờ khi biết cả hai biến cố cùng xảy ra phải bằng tổng lượng bất ngờ của từng biến cố riêng lẻ hợp lại.

        $$\text{Surprise}(E_1 \cap E_2) = \text{Surprise}(E_1) + \text{Surprise}(E_2)$$

        - **Nhưng thực tế**:
          - Theo xác suất, vì $E_1$ và $E_2$ độc lập nên:

            $$P(E_1 \cap E_2) = P(E_1) \times P(E_2)$$

          - Áp dụng công thức $\text{Surprise}(E) = \frac{1}{P(E)}$ vào $E_1 \cap E_2$:

          $$\text{Surprise}(E_1 \cap E_2) = \frac{1}{P(E_1 \cap E_2)} = \frac{1}{P(E_1) \times P(E_2)}$$

          - Tách phân số ra:

          $$\frac{1}{P(E_1) \times P(E_2)} = \frac{1}{P(E_1)} \times \frac{1}{P(E_2)}$$

          - Kết quả thực tế thu được là: $$\text{Surprise}(E_1 \cap E_2) = \text{Surprise}(E_1) \times \text{Surprise}(E_2)$$

        - Chúng ta mong muốn là **cộng** nhưng thực tế lại là **nhân**.

- Để giải quyết vấn đề này, các nhà toán học (điển hình là Claude Shannon) đã chèn thêm hàm **Logarithm** vào công thức tính **độ ngạc nhiên (surprise)** gọi là $I(p)$.
- $I(p)$ chính là cách ký hiệu toán học đại diện cho Surprise(E) (Độ bất ngờ / Hàm lượng thông tin) của một biến cố $E$ có xác suất xuất hiện là $p$.

- Trong **Lý thuyết thông tin (Information Theory)**:
  - $I$ là viết tắt của **Information (Thông tin)** hoặc **Self-Information (Tự thông tin)**.
  - $p$ là xác suất $P(E)$ xảy ra biến cố đó ($0 \le p \le 1$). Thay vì viết $I(P(E))$, người ta sẽ viết gọn lại thành $I(p)$.

  $$\text{Surprise}(E) = I(P(E)) = I(p) = \log_2\left(\frac{1}{P(E)}\right) = -\log_2(P(E))$$

- Với $E_1, E_2$ độc lập, thỏa phép cộng dồn:
$$I(E_1 \cap E_2) = -\log_2 [P(E_1)P(E_2)] = I(E_1) + I(E_2)$$

### 2.1. Bản chất cốt lõi: Entropy $H(S)$ thực chất là gì?

- **Entropy $H(S)$** chính là **giá trị kỳ vọng (trung bình trọng số)** độ bất ngờ $I(p_c)$ của tất cả các nhãn $c$ có trong tập dữ liệu $S$.
- **Entropy** càng lớn thì tập dữ liệu càng “không thuần”, càng khó phân tách.
  > $I(p_c)$ và $I(p_E)$ đều cùng là một khái niệm, chỉ khác nhau ở ký hiệu biểu diễn đối tượng biến cố/nhãn.
  
  
  $$H(S) = -\sum_{c \in \mathcal{C}} p_c \log_2 p_c \quad (\text{đơn vị: bit})$$

- Trong đó:
  - $S$ (Dataset): Tập dữ liệu cần đo độ hỗn loạn/độ xáo trộn.
  - $\mathcal{C}$ (Classes): Tập hợp tất cả các nhãn phân loại có thể có trong bài toán (Ví dụ: $\mathcal{C} = \{\text{Chơi}, \text{Không chơi}\}$ hoặc $\mathcal{C} = \{0, 1\}$).
  - $c$: Một nhãn phân loại cụ thể thuộc $\mathcal{C}$.
  - $p_c$: Xác suất xuất hiện của nhãn $c$ trong tập dữ liệu $S$.
  - $-\log_2 p_c$: Chính là $I(p_c)$ — lượng thông tin hay độ ngạc nhiên (Surprise) thu được khi xuất hiện nhãn $c$.
  - **Dấu $\sum$ (Tổng)**: Lấy tổng độ ngạc nhiên của từng nhãn.
  - **Đơn vị bit**: Do sử dụng hàm logarithm cơ số 2 ($\log_2$), đơn vị đo thông tin thu được tính bằng **bit**.

    

### 2.2. Ví dụ minh họa bằng số cụ thể về Entropy $H(S)$

- Giả sử bạn có tập dữ liệu $S$ gồm 10 mẫu phân loại thành 2 nhãn $\{A, B\}$:

#### 2.2.1. Trường hợp 1: Tập dữ liệu hoàn toàn tinh khiết (Pure)

- Gồm **10** mẫu nhãn **A**: $\rightarrow p_A = \frac{10}{10} = 1$
- **0** mẫu nhãn **B**:  $\rightarrow p_B = \frac{0}{10} = 0$.

$$H(S) = -\left(1 \cdot \log_2(1) + 0 \cdot \log_2(0)\right) = -(0 + 0) = 0 \text{ bit}$$

- **Ý nghĩa**: Entropy = 0 $\rightarrow$ Tập dữ liệu sạch tuyệt đối, không có sự xáo trộn hay mơ hồ nào

#### 2.2.2. Trường hợp 2: Tập dữ liệu xáo trộn tối đa (Impure / Uncertain)

- Gồm **5** mẫu nhãn **A**: $\rightarrow p_A = \frac{5}{10} = 0.5$
- **5** mẫu nhãn **B**:  $\rightarrow p_B = \frac{5}{10} = 0.5$.

$$H(S) = -\left(0.5 \cdot \log_2(0.5) + 0.5 \cdot \log_2(0.5)\right) = -\left(0.5 \cdot (-1) + 0.5 \cdot (-1)\right) = 1 \text{ bit}$$

- **Ý nghĩa**: Entropy đạt giá trị cực đại (= 1 với bài toán 2 nhãn) $\rightarrow$ Tập dữ liệu vô cùng xáo trộn, dự đoán hoàn toàn ngẫu nhiên


#### 2.2.3. Trường hợp 3: Bài toán có 3 nhãn (ví dụ: $A, B, C$)

- Giả sử tập dữ liệu $S$ có **10 mẫu**, được phân thành 3 nhãn $A, B, C$ với số lượng lần lượt là:
  - Nhãn $A$: 5 mẫu
  - Nhãn $B$: 3 mẫu
  - Nhãn $C$: 2 mẫu

  

- $S$ (Dataset): Là toàn bộ tập dữ liệu gồm **10 mẫu** mà ta đang xét.
- $\mathcal{C}$ (Classes): Là tập hợp tất cả các nhãn phân loại có trong tập $S$, cụ thể ở đây $\mathcal{C} = \{A, B, C\}$
- $c$: Là một nhãn cụ thể thuộc tập $\mathcal{C}$. Trong Trường hợp 3, $c$ sẽ lần lượt nhận từng giá trị là $A$, $B$, hoặc $C$ khi lấy tổng.
- $p_c$: Là xác suất xuất hiện của nhãn $c$ trong tập dữ liệu $S$. Cụ thể:
$$p_c = \frac{\text{Số phần tử có nhãn } c}{\text{Tổng số phần tử trong tập } S}$$
  - Khi $c = A \rightarrow p_A = \frac{5}{10} = 0.5$
  - Khi $c = B \rightarrow p_B = \frac{3}{10} = 0.3$
  - Khi $c = C \rightarrow p_C = \frac{2}{10} = 0.2$

- $-\log_2 p_c$: Là **độ ngạc nhiên (Surprise) / lượng thông tin $I(p_c)$** thu được nếu rút ngẫu nhiên được một mẫu mang nhãn $c$:
  - Khi $c = A \rightarrow  I(p_A) = -\log_2(0.5) = 1 \text{ bit}$
  - Khi $c = B \rightarrow  I(p_B) = -\log_2(0.3) \approx 1.73696 \text{ bit}$
  - Khi $c = C \rightarrow  I(p_C) = -\log_2(0.2) \approx 2.32193 \text{ bit}$


- Tính Entropy $H(S)$ bằng cách khai triển đầy đủ từng phần tử
  $$H(S) = \left[ p_A \times I(p_A) \right] + \left[ p_B \times I(p_B) \right] + \left[ p_C \times I(p_C) \right]$$

- Thay các giá trị vừa tính vào:
  $$H(S) = (0.5 \times 1) + (0.3 \times 1.73696) + (0.2 \times 2.32193)$$$$H(S) = 0.5 + 0.521088 + 0.464386$$$$H(S) \approx 1.48547 \text{ bit}$$

-  $H(S)=0$ khi S chỉ có 1 lớp, được gọi là thuần (Pure)
-  $H(S)=1$ đạt cực đại khi $S$ có 2 nhãn phân phối đều (phân loại nhị phân).
- Với $C >2$, tức nhiều hơn 2 nhãn mà dữ liệu phân phối đều thì $H(S) = log2|C|$:
  - $\vert{}\mathcal{C}\vert{}$ (hoặc $K$): Số lượng nhãn phân loại trong tập dữ liệu (ví dụ: có 3 nhãn thì $\vert{}\mathcal{C}\vert{} = 3$, có 4 nhãn thì $\vert{}\mathcal{C}\vert{} = 4$)
  - Dữ liệu phân phối đều: Tất cả $\vert{}\mathcal{C}\vert{}$ nhãn đều có số lượng bằng nhau, tức là cơ hội xuất hiện của mỗi nhãn là như nhau.
  
  $$p_c = \frac{1}{\vert{}\mathcal{C}\vert{}} \quad \text{cho mọi nhãn } c$$

- Thay vào: $$H(S) = -\sum_{c \in \mathcal{C}} p_c \log_2 p_c$$
  - Vì tất cả $\vert{}\mathcal{C}\vert{}$ nhãn đều có $p_c = \frac{1}{\vert{}\mathcal{C}\vert{}}$, ta thay giá trị này vào công thức:

    1. Thay $p_c$ vào:
      
      $$H(S) = -\sum_{c \in \mathcal{C}} \left( \frac{1}{\vert{}\mathcal{C}\vert{}} \cdot \log_2\left(\frac{1}{\vert{}\mathcal{C}\vert{}}\right) \right)$$
    
    2. Áp dụng tính chất **Logarithm** $\log_2\left(\frac{1}{x}\right) = -\log_2(x)$:
    
      $$\log_2\left(\frac{1}{\vert{}\mathcal{C}\vert{}}\right) = -\log_2(\vert{}\mathcal{C}\vert{})$$
    
    3. Thay ngược lại vào tổng:
      
      $$H(S) = -\sum_{c \in \mathcal{C}} \left( \frac{1}{\vert{}\mathcal{C}\vert{}} \cdot \left(-\log_2(\vert{}\mathcal{C}\vert{})\right) \right)$$
      
      $$H(S) = \sum_{c \in \mathcal{C}} \left( \frac{1}{\vert{}\mathcal{C}\vert{}} \cdot \log_2(\vert{}\mathcal{C}\vert{}) \right)$$
    
    4. Lấy tổng của $\vert{}\mathcal{C}\vert{}$ phần tử giống hệt nhau: Do ta đang cộng đại lượng $\left( \frac{1}{\vert{}\mathcal{C}\vert{}} \cdot \log_2(\vert{}\mathcal{C}\vert{}) \right)$ đúng $\vert{}\mathcal{C}\vert{}$ lần:
    
      $$H(S) = \vert{}\mathcal{C}\vert{} \times \left( \frac{1}{\vert{}\mathcal{C}\vert{}} \cdot \log_2(\vert{}\mathcal{C}\vert{}) \right)$$
      
    5. Rút gọn $\vert{}\mathcal{C}\vert{}$, ta còn lại:

      $$H(S) = \log_2(\vert{}\mathcal{C}\vert{})$$

- Nếu có 2 nhãn phân phối đều ($p_1 = p_2 = 0.5$):$$H(S) = \log_2(2) = 1 \text{ bit}$$

- Nếu có 3 nhãn phân phối đều ($p_1 = p_2 = p_3 = \frac{1}{3}$):$$H(S) = \log_2(3) \approx 1.585 \text{ bit}$$

- Nếu có 4 nhãn phân phối đều ($p_1 = p_2 = p_3 = p_4 = 0.25$):$$H(S) = \log_2(4) = 2 \text{ bit}$$

  > **Đây chính là Entropy cực đại ($H_{\max}$)**

#### 2.2.4. Đồ thị của hàm Entropy:
- Đồ thị của hàm **Entropy** mô tả mối quan hệ giữa xác suất xuất hiện của nhãn ($p$) và độ xáo trộn/bất định ($H(S)$).Để dễ hình dung nhất, ta xét đồ thị **Entropy** cho bài toán nhị phân (2 nhãn $A$ và $B$), với xác suất xuất hiện nhãn $A$ là $p$, và nhãn $B$ là $1 - p$.

- **Công thức đường cong Entropy nhị phân**:

    $$H(p) = -p \log_2(p) - (1-p) \log_2(1-p)$$

  - **Trục hoành (Trục X)**: Biểu diễn xác suất $p$ (chạy từ $0.0$ đến $1.0$).
  - **Trục tung (Trục Y)**: Biểu diễn giá trị Entropy $H(p)$ tính bằng bit (chạy từ $0.0$ đến $1.0$)


<p align="center">
  <img src="../Images/e0cbfe2d-9702-427d-aaf9-a142635062ec.png" alt="e0cbfe2d-9702-427d-aaf9-a142635062ec"/>
</p>

- **Các điểm quan trọng trên đường cong:**
  - Tại $p = 0$ ($0\%$ mẫu nhãn $A$, $100\%$ mẫu nhãn $B$):
    - $H(p) = 0\text{ bit}$
    - Tập dữ liệu thuần khiết tuyệt đối, không có sự xáo trộn nào.
  - Tại $p = 1$ ($100\%$ mẫu nhãn $A$, $0\%$ mẫu nhãn $B$):
    - $H(p) = 0\text{ bit}$
    - Tập dữ liệu thuần khiết tuyệt đối, không có sự xáo trộn nào.
  - Tại $p = 0.5$ ($50\%$ mẫu nhãn $A$, $50\%$ mẫu nhãn $B$):
    - $H(p) = 1\text{ bit}$ (Đỉnh của đường cong)
    - Dữ liệu phân phối đều 100%, độ ngạc nhiên/không chắc chắn đạt mức tối đa.